# FIP+eDAS SAVE 数据读取与绘图

本 Notebook 用于读取 Tab3 `FIP+eDAS SAVE` 生成的联合存储 `.npz` 文件，输出 FIP 与 eDAS 的采集参数，并绘制 FIP 时域、FIP PSD、eDAS Space-Time、指定 eDAS 通道时域和指定通道 PSD。

联合存储文件由 `src/das_tab3/das_storage_worker.py` 写入，主要字段包括 `comm_counts`、`packet_start_times`、`packet_duration_seconds`、`fip_present`、`das_present`、`fip_raw_200khz`、`fip_display_data`、`das_raw_matrix`，新版本还包含 `fip_sample_rate_hz`、`das_sample_rate_hz` 和 `das_channel_count`。

Update: joint v3 files are read with FIP1/FIP2-aware fields first (`fip1_raw_200khz`, `fip2_raw_200khz`, `fip1_display_data`, `fip2_display_data`, `fip_sensor_count`, `fip_selected_sensor`). Legacy fields `fip_raw_200khz` and `fip_display_data` are still used as fallback. The `200khz` suffix is historical; use `fip_sample_rate_hz` as the actual sample rate.


In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt

# 修改为需要读取的 FIP+eDAS SAVE 文件路径。
DATA_FILE = Path(r"D:/PCCP/FIPeDASDATA/FIPeDAS-YYYYMMDD-HHMMSS.mmm.npz")

# Select FIP sensor for plotting: 1 = FIP1, 2 = FIP2. Legacy single-FIP files fall back automatically.
FIP_SENSOR_TO_PLOT = 1

# eDAS 指定通道，按 0 起始索引。
EDAS_CHANNEL_INDEX = 0

# Space-Time 显示范围，默认与 Tab3 当前默认色标一致。
SPACE_TIME_VMIN = -0.3
SPACE_TIME_VMAX = 0.3

# 绘图降采样上限，防止一次绘制过多点导致 Notebook 卡顿。
MAX_PLOT_POINTS = 300_000
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
def load_joint_npz(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with np.load(path, allow_pickle=True) as npz:
        return {key: npz[key] for key in npz.files}


def object_array_to_list(value) -> list[np.ndarray]:
    if value is None:
        return []
    arr = np.asarray(value, dtype=object)
    result = []
    for item in arr:
        if item is None:
            result.append(np.array([], dtype=np.float64))
        else:
            result.append(np.asarray(item, dtype=np.float64))
    return result


def has_nonempty_frame(frames) -> bool:
    return any(np.asarray(frame).size > 0 for frame in frames)


def choose_frames(primary, fallback=None):
    if has_nonempty_frame(primary):
        return primary
    if fallback is not None and has_nonempty_frame(fallback):
        return fallback
    return primary


def finite_median(values, default=np.nan):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return default
    return float(np.median(values))


def decimate_for_plot(x, y, max_points=MAX_PLOT_POINTS):
    x = np.asarray(x)
    y = np.asarray(y)
    if x.size <= max_points:
        return x, y
    step = int(math.ceil(x.size / max_points))
    return x[::step], y[::step]


def compute_psd(signal, sample_rate_hz):
    signal = np.asarray(signal, dtype=np.float64)
    signal = signal[np.isfinite(signal)]
    if signal.size < 8 or not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0:
        return np.array([]), np.array([])
    signal = signal - np.mean(signal)
    try:
        from scipy.signal import welch
        nperseg = min(8192, signal.size)
        return welch(signal, fs=float(sample_rate_hz), nperseg=nperseg)
    except Exception:
        window = np.hanning(signal.size)
        spectrum = np.fft.rfft(signal * window)
        freqs = np.fft.rfftfreq(signal.size, d=1.0 / float(sample_rate_hz))
        scale = float(sample_rate_hz) * np.sum(window ** 2)
        psd = (np.abs(spectrum) ** 2) / max(scale, 1e-30)
        return freqs, psd


In [ ]:
data = load_joint_npz(DATA_FILE)
print("Fields:", sorted(data.keys()))

comm_counts = np.asarray(data.get("comm_counts", []), dtype=np.int64)
packet_start_times = np.asarray(data.get("packet_start_times", []), dtype=np.float64)
packet_duration_seconds = np.asarray(data.get("packet_duration_seconds", []), dtype=np.float64)
fip_present = np.asarray(data.get("fip_present", np.ones_like(comm_counts, dtype=bool)), dtype=bool)
das_present = np.asarray(data.get("das_present", np.ones_like(comm_counts, dtype=bool)), dtype=bool)

fip_compat_raw_frames = object_array_to_list(data.get("fip_raw_200khz"))
fip_compat_display_frames = object_array_to_list(data.get("fip_display_data"))
fip1_raw_frames = choose_frames(object_array_to_list(data.get("fip1_raw_200khz")), fip_compat_raw_frames)
fip2_raw_frames = object_array_to_list(data.get("fip2_raw_200khz"))
fip1_display_frames = choose_frames(object_array_to_list(data.get("fip1_display_data")), fip_compat_display_frames)
fip2_display_frames = object_array_to_list(data.get("fip2_display_data"))

fip_sensor_to_plot = 2 if int(FIP_SENSOR_TO_PLOT) == 2 else 1
if fip_sensor_to_plot == 2:
    fip_raw_frames = choose_frames(fip2_raw_frames, fip_compat_raw_frames)
    fip_display_frames = choose_frames(fip2_display_frames, fip_compat_display_frames)
else:
    fip_raw_frames = fip1_raw_frames
    fip_display_frames = fip1_display_frames

das_frames = object_array_to_list(data.get("das_raw_matrix"))

fip_rates = np.asarray(data.get("fip_sample_rate_hz", np.full(comm_counts.shape, np.nan)), dtype=np.float64)
das_rates = np.asarray(data.get("das_sample_rate_hz", np.full(comm_counts.shape, np.nan)), dtype=np.float64)
das_channel_counts = np.asarray(data.get("das_channel_count", np.zeros(comm_counts.shape, dtype=np.int32)), dtype=np.int32)
fip_sensor_counts = np.asarray(data.get("fip_sensor_count", np.zeros(comm_counts.shape, dtype=np.int32)), dtype=np.int32)
fip_selected_sensors = np.asarray(data.get("fip_selected_sensor", np.zeros(comm_counts.shape, dtype=np.int32)), dtype=np.int32)

for idx, frame in enumerate(fip_raw_frames):
    if idx < fip_rates.size and not np.isfinite(fip_rates[idx]) and idx < packet_duration_seconds.size and packet_duration_seconds[idx] > 0 and frame.size > 0:
        fip_rates[idx] = frame.size / packet_duration_seconds[idx]
for idx, matrix in enumerate(das_frames):
    if idx < das_rates.size and not np.isfinite(das_rates[idx]) and idx < packet_duration_seconds.size and packet_duration_seconds[idx] > 0 and matrix.ndim == 2 and matrix.shape[1] > 0:
        das_rates[idx] = matrix.shape[1] / packet_duration_seconds[idx]
    if idx < das_channel_counts.size and das_channel_counts[idx] <= 0 and matrix.ndim == 2:
        das_channel_counts[idx] = matrix.shape[0]

summary = {
    "file": str(DATA_FILE),
    "format_version": str(data.get("format_version", "legacy")),
    "frame_count": int(comm_counts.size),
    "comm_count_range": None if comm_counts.size == 0 else (int(comm_counts[0]), int(comm_counts[-1])),
    "fip_present_frames": int(np.sum(fip_present)),
    "fip_sensor_count_median": int(finite_median(fip_sensor_counts[fip_sensor_counts > 0], default=1)),
    "file_selected_fip_median": int(finite_median(fip_selected_sensors[fip_selected_sensors > 0], default=1)),
    "notebook_plot_fip": f"FIP{fip_sensor_to_plot}",
    "fip1_nonempty_frames": sum(1 for frame in fip1_raw_frames if frame.size > 0),
    "fip2_nonempty_frames": sum(1 for frame in fip2_raw_frames if frame.size > 0),
    "edas_present_frames": int(np.sum(das_present)),
    "packet_duration_median_s": finite_median(packet_duration_seconds),
    "fip_sample_rate_median_hz": finite_median(fip_rates),
    "selected_fip_samples_first_valid_frame": next((int(frame.size) for frame in fip_raw_frames if frame.size > 0), 0),
    "edas_sample_rate_median_hz": finite_median(das_rates),
    "edas_channel_count_median": int(finite_median(das_channel_counts[das_channel_counts > 0], default=0)),
    "edas_samples_per_channel_first_valid_frame": next((int(frame.shape[1]) for frame in das_frames if frame.ndim == 2 and frame.size > 0), 0),
}
for key, value in summary.items():
    print(f"{key}: {value}")


In [ ]:
def concatenate_fip(frames, rates, starts, durations, present_mask):
    times = []
    values = []
    sample_rates = []
    for idx, frame in enumerate(frames):
        if idx >= len(present_mask) or not present_mask[idx] or frame.size == 0:
            continue
        duration = float(durations[idx]) if idx < len(durations) and durations[idx] > 0 else np.nan
        rate = float(rates[idx]) if idx < len(rates) and np.isfinite(rates[idx]) and rates[idx] > 0 else np.nan
        if not np.isfinite(rate) and np.isfinite(duration) and duration > 0:
            rate = frame.size / duration
        if not np.isfinite(rate) or rate <= 0:
            continue
        start = float(starts[idx]) if idx < len(starts) else 0.0
        times.append(start + np.arange(frame.size, dtype=np.float64) / rate)
        values.append(frame)
        sample_rates.append(rate)
    if not values:
        return np.array([]), np.array([]), np.nan
    return np.concatenate(times), np.concatenate(values), finite_median(sample_rates)

fip_source_frames = fip_raw_frames if has_nonempty_frame(fip_raw_frames) else fip_display_frames
fip_time, fip_values, fip_sample_rate = concatenate_fip(
    fip_source_frames,
    fip_rates,
    packet_start_times,
    packet_duration_seconds,
    fip_present,
)

if fip_values.size == 0:
    print(f"No drawable FIP{fip_sensor_to_plot} data.")
else:
    x_plot, y_plot = decimate_for_plot(fip_time, fip_values)
    plt.figure(figsize=(12, 4))
    plt.plot(x_plot, y_plot, linewidth=0.8)
    plt.title(f"FIP{fip_sensor_to_plot} Time Domain")
    plt.xlabel("Time (s)")
    plt.ylabel("Phase / amplitude")
    plt.tight_layout()
    plt.show()

    freqs, psd = compute_psd(fip_values, fip_sample_rate)
    if freqs.size:
        plt.figure(figsize=(12, 4))
        plt.semilogy(freqs, psd, linewidth=0.9)
        plt.title(f"FIP{fip_sensor_to_plot} PSD")
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("PSD")
        plt.tight_layout()
        plt.show()


In [ ]:
def valid_das_frames(frames, present_mask):
    selected = []
    selected_indices = []
    reference_shape = None
    for idx, matrix in enumerate(frames):
        if idx >= len(present_mask) or not present_mask[idx] or matrix.ndim != 2 or matrix.size == 0:
            continue
        if reference_shape is None:
            reference_shape = matrix.shape
        if matrix.shape != reference_shape:
            print(f"跳过 shape 不一致的 eDAS 帧 idx={idx}, shape={matrix.shape}, reference={reference_shape}")
            continue
        selected.append(matrix)
        selected_indices.append(idx)
    return selected, selected_indices

das_valid, das_indices = valid_das_frames(das_frames, das_present)
if not das_valid:
    print("没有可绘制的 eDAS 数据。")
else:
    space_time_matrix = np.concatenate(das_valid, axis=1)
    start_time = float(packet_start_times[das_indices[0]]) if len(packet_start_times) else 0.0
    last_idx = das_indices[-1]
    end_time = float(packet_start_times[last_idx] + packet_duration_seconds[last_idx]) if len(packet_start_times) and len(packet_duration_seconds) else space_time_matrix.shape[1]
    plt.figure(figsize=(12, 5))
    plt.imshow(
        space_time_matrix,
        aspect="auto",
        origin="lower",
        extent=[start_time, end_time, 0, space_time_matrix.shape[0] - 1],
        cmap="seismic",
        vmin=SPACE_TIME_VMIN,
        vmax=SPACE_TIME_VMAX,
    )
    plt.colorbar(label="eDAS value")
    plt.title("eDAS Space-Time")
    plt.xlabel("Time (s)")
    plt.ylabel("Channel")
    plt.tight_layout()
    plt.show()


In [ ]:
def concatenate_edas_channel(frames, indices, channel_index, starts, durations, rates):
    times = []
    values = []
    sample_rates = []
    for matrix, idx in zip(frames, indices):
        if channel_index < 0 or channel_index >= matrix.shape[0]:
            continue
        y = matrix[channel_index]
        duration = float(durations[idx]) if idx < len(durations) and durations[idx] > 0 else np.nan
        rate = float(rates[idx]) if idx < len(rates) and np.isfinite(rates[idx]) and rates[idx] > 0 else np.nan
        if not np.isfinite(rate) and np.isfinite(duration) and duration > 0:
            rate = y.size / duration
        if not np.isfinite(rate) or rate <= 0:
            continue
        start = float(starts[idx]) if idx < len(starts) else 0.0
        times.append(start + np.arange(y.size, dtype=np.float64) / rate)
        values.append(y)
        sample_rates.append(rate)
    if not values:
        return np.array([]), np.array([]), np.nan
    return np.concatenate(times), np.concatenate(values), finite_median(sample_rates)

if not das_valid:
    print("没有可绘制的 eDAS 单通道数据。")
else:
    edas_time, edas_values, edas_sample_rate = concatenate_edas_channel(
        das_valid,
        das_indices,
        EDAS_CHANNEL_INDEX,
        packet_start_times,
        packet_duration_seconds,
        das_rates,
    )
    if edas_values.size == 0:
        print(f"通道 {EDAS_CHANNEL_INDEX} 不存在或无有效数据。")
    else:
        x_plot, y_plot = decimate_for_plot(edas_time, edas_values)
        plt.figure(figsize=(12, 4))
        plt.plot(x_plot, y_plot, linewidth=0.8)
        plt.title(f"eDAS Channel {EDAS_CHANNEL_INDEX} 时域")
        plt.xlabel("Time (s)")
        plt.ylabel("eDAS value")
        plt.tight_layout()
        plt.show()

        freqs, psd = compute_psd(edas_values, edas_sample_rate)
        if freqs.size:
            plt.figure(figsize=(12, 4))
            plt.semilogy(freqs, psd, linewidth=0.9)
            plt.title(f"eDAS Channel {EDAS_CHANNEL_INDEX} PSD")
            plt.xlabel("Frequency (Hz)")
            plt.ylabel("PSD")
            plt.tight_layout()
            plt.show()
